# ขั้นที่ 2: Forecast — พยากรณ์ยอดขาย 28 วันข้างหน้า

## ภาพรวมของขั้นนี้

| | |
|---|---|
| **คำถาม** | อีก 28 วันข้างหน้า สินค้าแต่ละรายการในแต่ละสาขาจะขายได้กี่ชิ้นต่อวัน |
| **ทำไมต้องรู้** | ใช้ตัดสินใจว่าจะสั่งของเท่าไร ถ้าทายต่ำไปของขาด ถ้าทายสูงไปของล้น (ผลกระทบเป็นเงินดูในขั้นที่ 3) |
| **ขนาดงาน** | 30,490 รายการ (สินค้า × สาขา) × 28 วัน = พยากรณ์ 853,720 ค่า |

## ขั้นตอนการทำงาน

```
2.1 วิธีวัดผล         ใช้ WRMSSE + ทดสอบย้อนหลัง 3 รอบ (backtest)
      ↓
2.2 Feature engineering   แปลงข้อมูลดิบเป็นตัวแปรที่โมเดลใช้ได้ (จาก insight ขั้นที่ 1)
      ↓
2.3 Baseline           วิธีง่ายๆ ที่โมเดลต้องเอาชนะให้ได้
      ↓
2.4 โมเดล ML            LightGBM, XGBoost, CatBoost, Random Forest, LightGBM รายสาขา
      ↓
2.5 Backtest           เทรนและวัดผลทุกวิธีบนข้อมูลชุดเดียวกัน
      ↓
2.6 ผลลัพธ์             ภาพรวม → แยกหมวด → แยกระดับ → รายสินค้า
      ↓
2.7 พยากรณ์จริง         28 วันข้างหน้า (23 พ.ค. – 19 มิ.ย. 2016)
      ↓
2.8 อธิบายโมเดล (XAI)    ปัจจัยไหนสำคัญที่สุด และทำไมทายเท่านี้ (SHAP)
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True

FIGURES = ROOT / 'reports' / 'figures'
ANALYSIS = ROOT / 'analysis'
FIGURES.mkdir(parents=True, exist_ok=True)
ANALYSIS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)


def show(fig, name):
    """แสดงกราฟในหน้านี้ และบันทึกสำเนาไว้ใน reports/figures"""
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
import matplotlib.dates as mdates
import shap

from src.data import PROCESSED, ID_COLS, load_long, load_raw
from src.features import CAT_FEATURES, HORIZON, FeatureBuilder
from src.metrics import WRMSSE
from src import backtest as bt

In [ ]:
# ค่าตั้งต้นของ backtest อยู่ใน src/backtest.py (ใช้ร่วมกับคำสั่งรันทีละโมเดล)
FOLDS, FINAL_END, TRAIN_DAYS, STORE_TRAIN_DAYS = bt.FOLDS, bt.FINAL_END, bt.TRAIN_DAYS, bt.STORE_TRAIN_DAYS
rng = np.random.default_rng(42)

sales, calendar, prices = load_raw()
fb = FeatureBuilder(load_long(), calendar)
Y = fb.sales  # ตารางยอดขาย (สินค้า×สาขา, วัน) คอลัมน์ j คือวัน d_(j+1)
d_cols = [c for c in sales.columns if c.startswith('d_')]
dates = pd.to_datetime(calendar['date']).to_numpy()
print(f'ยอดขาย {Y.shape[0]:,} รายการ × {Y.shape[1]:,} วัน')

---
## 2.1 วิธีวัดผล

### ตัวชี้วัด: WRMSSE (ยิ่งต่ำยิ่งดี)

อธิบายแบบง่ายใน 3 ข้อ
1. **เทียบกับวิธีง่ายที่สุด:** วัด error ของการพยากรณ์ หารด้วย error ของวิธี "ทายว่าวันนี้ขายเท่าเมื่อวาน" ตัวหารคำนวณจากความคลาดเคลื่อนของ naive ในช่วง train ค่า 1 จึงไม่ใช่คะแนนของ baseline ในช่วงทดสอบ ต้องเปรียบเทียบกับคะแนน baseline ที่วัดจริง
2. **ให้น้ำหนักตามรายได้:** สินค้าที่ทำเงินมากมีผลต่อคะแนนมาก เพราะถ้าทายผิดจะเสียหายมาก
3. **วัดทุกระดับ:** วัดทั้งยอดรวมบริษัท รายรัฐ รายสาขา รายหมวด ... ลงไปถึงรายสินค้า×สาขา รวม 12 ระดับ แล้วเฉลี่ย โมเดลจึงต้องแม่นทั้งภาพรวมและรายตัว

### ทดสอบย้อนหลัง (backtest)

แกล้งทำเป็นว่าอยู่ในอดีต เทรนโมเดลด้วยข้อมูลถึงวันหนึ่ง แล้วทาย 28 วันถัดไปซึ่งรู้คำตอบจริงอยู่แล้ว
ทดสอบ 3 รอบติดกัน รอบละ 28 วัน (d_1858–d_1885, d_1886–d_1913, d_1914–d_1941) แล้วเลือกโมเดลจาก **ค่าเฉลี่ย** และดู **std** ว่าผลสม่ำเสมอแค่ไหนในแต่ละช่วงเวลา **ห้ามสุ่มแบ่งข้อมูล** เพราะจะกลายเป็นใช้อนาคตทายอดีต

In [ ]:
fig, ax = plt.subplots(figsize=(13, 2.8))
rows = [(f'Backtest {i + 1}', end, 'darkorange') for i, end in enumerate(FOLDS)] + [('Final forecast', FINAL_END, 'seagreen')]
day_num = mdates.date2num(dates)  # วันที่ในหน่วย "วัน" ของ matplotlib ใช้กำหนดความกว้างแท่งได้
for i, (label, end, color) in enumerate(rows):
    ax.barh(i, TRAIN_DAYS, left=day_num[end - TRAIN_DAYS], color='steelblue')
    ax.barh(i, HORIZON, left=day_num[end], color=color)
    ax.text(day_num[end + HORIZON - 1] + 4, i, f'{"forecast" if end == FINAL_END else "test"} {pd.Timestamp(dates[end]):%d %b} – {pd.Timestamp(dates[end + HORIZON - 1]):%d %b %Y}', va='center')
ax.set_yticks(range(len(rows)), [r[0] for r in rows])
ax.invert_yaxis()
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in ['steelblue', 'darkorange', 'seagreen']],
          labels=['Train (365 days; 730 for per-store model)', 'Test (28 days, actual known)', 'Forecast (future, unknown)'], loc='lower left', fontsize=8)
ax.xaxis_date()
ax.set(title='Backtest design', xlim=(day_num[FOLDS[0] - TRAIN_DAYS - 10], day_num[-1] + 75))
show(fig, '02_1_backtest_design')

---
## 2.2 Feature engineering

โมเดลไม่สามารถอ่าน "ตารางยอดขาย" ได้โดยตรง ต้องแปลงเป็นตัวแปร (feature) ที่บอกข้อมูลแต่ละวันของแต่ละสินค้า
feature ทุกตัวมาจาก insight ในขั้นที่ 1

| กลุ่ม | Feature | ความหมาย | มาจาก insight |
|---|---|---|---|
| **ยอดขายในอดีต (lag)** | `lag_28`, `lag_35`, ..., `lag_56` | ยอดขายเมื่อ 28, 35, ..., 56 วันก่อน | ยอดขายมีรูปแบบซ้ำรายสัปดาห์ (1.1) |
| **ค่าเฉลี่ยเคลื่อนที่ (rolling)** | `rmean_7`, `rmean_28`, `rmean_56` | ยอดขายเฉลี่ย 7/28/56 วัน นับถอยหลังจาก 28 วันก่อน | บอกระดับยอดขายปกติและแนวโน้ม (1.1, 1.5) |
| **ราคา** | `sell_price`, `price_rel_regular`, `price_change_7d` | ราคาปัจจุบัน, ราคาเทียบราคาสูงสุดที่ทราบจนถึงวันนั้น, ราคาเปลี่ยนจากสัปดาห์ก่อนกี่ % | ลดราคาแล้วขายดีขึ้น (1.4.4) |
| **ปฏิทิน** | `wday`, `month`, `mday` | วันในสัปดาห์, เดือน, วันที่ของเดือน | เสาร์-อาทิตย์ขายดี (1.4.1) |
| **Event / SNAP** | `event_name_1`, `event_type_1`, `snap` | วันนั้นมีเทศกาลอะไร, เป็นวัน SNAP ของรัฐนั้นไหม | ยอดเปลี่ยนในวัน event และวัน SNAP (1.4.2–1.4.3) |
| **รหัสสินค้า/สถานที่** | `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` | สินค้าอะไร อยู่สาขาไหน | แต่ละสาขาและหมวดขายต่างกัน (1.2–1.3) |

### ทำไม lag ต้องเริ่มที่ 28 วัน

ต้องการทาย 28 วันข้างหน้า **ในครั้งเดียว** ถ้าใช้ยอดขาย "เมื่อวาน" เป็น feature จะทายได้แค่วันแรก เพราะวันที่ 2–28 ยังไม่รู้ยอดของเมื่อวาน
การใช้ข้อมูลที่เก่ากว่า 28 วันทำให้ทุกวันในช่วงพยากรณ์มี feature ครบ เรียกวิธีนี้ว่า **direct forecasting** ข้อดีคือ error ไม่สะสม

**สมมติฐานเรื่องราคา:** การพยากรณ์ใช้ราคาของแต่ละวันในช่วง 28 วันข้างหน้าจาก dataset จึงสมมติว่าทราบราคาล่วงหน้า ราคาสูงสุดอ้างอิงใช้เฉพาะข้อมูลถึงวันนั้น

ตัวอย่างตาราง feature ที่โมเดลเห็น (1 แถว = สินค้า 1 รายการ ใน 1 สาขา ใน 1 วัน)

In [ ]:
d_from = FINAL_END - 179
window = slice(d_from - 1, FINAL_END)
with np.errstate(invalid='ignore'):
    price_changed = np.nanmax(fb.price[:, window], axis=1) > np.nanmin(fb.price[:, window], axis=1) * 1.02
revenue = (Y[:, window] * np.nan_to_num(fb.price[:, window])).sum(axis=1)
top_idx = int(np.argmax(np.where(price_changed, revenue, -1)))  # รายได้สูงสุดในกลุ่มที่มีการเปลี่ยนราคา
top_name = f"{sales.loc[top_idx, 'item_id']} @ {sales.loc[top_idx, 'store_id']}"
X_demo, y_demo = fb.build(d_from, FINAL_END)
one = X_demo.iloc[top_idx * 180:(top_idx + 1) * 180].assign(sales=y_demo[top_idx * 180:(top_idx + 1) * 180])
one.index = pd.DatetimeIndex(dates[d_from - 1:FINAL_END], name='date')
print(f'ตัวอย่าง: {top_name} (รายได้สูงสุดในกลุ่มสินค้าที่มีการเปลี่ยนราคาใน 180 วัน) | feature ทั้งหมด {X_demo.shape[1]} ตัว')
one.tail(5).T

ตัวอย่างการทำงานของ feature กับสินค้าตัวเดียวกัน (180 วันล่าสุด)
- **กราฟบน:** ยอดขายจริง (แท่ง) เทียบกับ `lag_28` และ `rmean_28` จะเห็นว่า rolling mean จับระดับยอดขายได้ดี ส่วน lag เก็บรูปแบบรายสัปดาห์ไว้
- **กราฟล่าง:** ราคาเทียบกับราคาปกติ (1.0 = ราคาปกติ, ต่ำกว่า 1 = ลดราคา) ให้สังเกตยอดขายในกราฟบนช่วงที่ราคาเปลี่ยน

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
axes[0].bar(one.index, one['sales'], color='lightgray', label='Actual sales')
axes[0].plot(one.index, one['lag_28'], color='darkorange', lw=1, label='lag_28 (sales 28 days ago)')
axes[0].plot(one.index, one['rmean_28'], color='steelblue', lw=2, label='rmean_28 (28-day avg, ending 28 days ago)')
axes[0].set(title=f'Sales features: {top_name}', ylabel='Units')
axes[0].legend(loc='upper left', fontsize=8)
axes[1].plot(one.index, one['price_rel_regular'], color='seagreen', drawstyle='steps-post')
axes[1].set(title='price_rel_regular (1.0 = regular price)', ylabel='Ratio')
show(fig, '02_2_feature_example')

**สิ่งที่เห็นจากตัวอย่างนี้**
- `rmean_28` ตามระดับยอดขายได้ดี แต่ช้ากว่ายอดจริง 28 วันตามที่ออกแบบไว้
- ช่วงต้นเดือนมีนาคม ยอดขายเป็น 0 ติดกันเกือบ 3 สัปดาห์ ทั้งที่ปกติขายได้วันละ 50–100 ชิ้น และตรงกับช่วงที่ลดราคา อาจเกิดจาก **ของหมดสต็อก** หรือสาเหตุอื่นได้ แต่ข้อมูลไม่มีสต็อกให้ยืนยัน (เป็นข้อสันนิษฐานเรื่อง censored demand)
- 28 วันต่อมา `lag_28` และ `rmean_28` ตกลงตาม โมเดลจึงอาจทายต่ำเกินไปในช่วงนั้น นี่คือข้อจำกัดของข้อมูลที่ไม่มีระดับสต็อก (กล่าวถึงอีกครั้งในขั้นที่ 4)

**feature ไหนสัมพันธ์กับยอดขายมากที่สุด**

ดูความสัมพันธ์แบบ Spearman ระหว่าง feature ตัวเลขแต่ละตัวกับยอดขายจริง (สุ่ม 300,000 แถว)
เป็นการตรวจสอบเบื้องต้นก่อนเทรนโมเดล ส่วนความสำคัญจริงของ feature ในโมเดลดูได้ที่หัวข้อ 2.7

In [ ]:
numeric = [c for c in X_demo.columns if c not in CAT_FEATURES]
sample_rows = rng.choice(len(X_demo), 300_000, replace=False)
corr = X_demo.iloc[sample_rows][numeric].reset_index(drop=True).corrwith(pd.Series(y_demo[sample_rows]), method='spearman').sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
corr.plot.barh(ax=ax, color=np.where(corr > 0, 'steelblue', 'indianred'))
ax.set(title='Spearman correlation of each feature with daily sales', xlabel='Correlation')
show(fig, '02_2_feature_correlation')
del X_demo, y_demo

**ข้อสังเกต:** feature ยอดขายในอดีต (rolling mean และ lag) สัมพันธ์กับยอดขายสูงที่สุด เพราะสินค้าที่เคยขายดีมักจะขายดีต่อ
ส่วนปฏิทิน ราคา และ SNAP สัมพันธ์น้อยกว่าเมื่อดูทีละตัว แต่ช่วยอธิบายว่า **วันไหน** จะขายมากหรือน้อยกว่าระดับปกติ

---
## 2.3 Baseline: วิธีง่ายๆ ที่โมเดลต้องเอาชนะ

| วิธี | ทายอย่างไร |
|---|---|
| Naive | ทุกวันขายเท่ากับวันสุดท้ายที่รู้ |
| Seasonal naive | ทำซ้ำยอดขาย 7 วันสุดท้าย (จับรูปแบบรายสัปดาห์) |
| Mean 28d | ค่าเฉลี่ย 28 วันล่าสุด |

ถ้าโมเดล ML ชนะ baseline ไม่ได้ ก็ไม่คุ้มที่จะใช้

In [ ]:
def naive(end):
    return np.repeat(Y[:, end - 1:end], HORIZON, axis=1)


def seasonal_naive(end):
    return np.tile(Y[:, end - 7:end], HORIZON // 7)


def mean_28d(end):
    return np.repeat(Y[:, end - 28:end].mean(axis=1, keepdims=True), HORIZON, axis=1)


BASELINES = {'Naive': naive, 'Seasonal naive': seasonal_naive, 'Mean 28d': mean_28d}

---
## 2.4 โมเดล Machine Learning

ใช้โมเดลแบบต้นไม้ตัดสินใจ (tree-based) 4 แบบ ทุกโมเดลใช้ feature ชุดเดียวกัน โมเดลหลักเป็น **โมเดลเดียวที่เรียนจากทุกสินค้าพร้อมกัน** (global model)
เพราะสินค้าส่วนใหญ่ขายช้า ถ้าแยกโมเดลรายสินค้าจะมีข้อมูลน้อยเกินไป ส่วน **LightGBM รายสาขา** แยกระดับกลาง คือ 1 โมเดลต่อสาขา

| โมเดล | หลักการ | เหตุผลที่เลือก |
|---|---|---|
| **LightGBM** | Gradient boosting: สร้างต้นไม้ทีละต้น แต่ละต้นแก้ error ของต้นก่อนหน้า | รองรับ categorical และ Tweedie ใช้ RAM น้อย (เวลาเทรนจริงดูในหัวข้อ 2.6) |
| **XGBoost** | Gradient boosting เหมือนกัน แต่วิธีสร้างต้นไม้ต่างกัน | แม่นและเสถียร ใช้เปรียบเทียบกับ LightGBM |
| **CatBoost** | Gradient boosting ที่จัดการตัวแปรประเภทหมวดหมู่ได้เอง | ไม่ต้องแปลงรหัสสินค้าและสาขาเอง |
| **Random Forest** | สร้างต้นไม้หลายต้นอิสระกันแล้วเฉลี่ยผล | เข้าใจง่าย ใช้เป็นตัวเทียบฝั่ง bagging |
| **LightGBM รายสาขา** | LightGBM ตั้งค่าเดียวกัน แต่แยก 1 โมเดลต่อสาขา (10 โมเดล) และใช้ประวัติ 2 ปี | แต่ละสาขามีรูปแบบการขายต่างกัน (1.2) และผู้ชนะอันดับ 1 ของ M5 ก็แยกโมเดลรายสาขา |

โมเดล boosting ใช้ **Tweedie loss** ซึ่งเหมาะกับยอดขายที่เป็น 0 บ่อย (จากหัวข้อ 1.5)
CatBoost และ Random Forest ช้ากว่ามาก จึงเทรนบนข้อมูลที่สุ่มมาบางส่วน

**การรันแบบแบ่ง batch:** โค้ดเทรนอยู่ใน `src/backtest.py` เพราะ 5 โมเดล × 3 fold ใช้เวลารวมหลายชั่วโมง จึงรันทีละโมเดลจาก terminal ได้ และผลของแต่ละ fold ถูกบันทึกใน `data/processed/backtest/` ทันทีที่เสร็จ หยุดกลางทางแล้วรันต่อได้

```
python -m src.backtest --list                 # ดูว่าโมเดลไหนรันเสร็จแล้วกี่ fold
python -m src.backtest "LightGBM"             # backtest 3 fold ของโมเดลนี้
python -m src.backtest "LightGBM" --final     # เทรนถึง d_1941 เพื่อพยากรณ์จริง
```

notebook นี้อ่านผลจากไฟล์ ถ้า fold ไหนยังไม่มีผลจะเทรนให้ตอนรัน

In [ ]:
MODELS = bt.MODEL_NAMES


def model_forecast(name, end):
    """อ่านผล backtest จากไฟล์ ถ้ายังไม่มีให้เทรนแล้วบันทึก คืนค่า (pred, เวลาเทรนเป็นนาที)"""
    cached = bt.load_fold(name, end)
    return cached if cached is not None else bt.run_fold(name, fb, end)


print('ค่าตั้งต้น:', {k: getattr(bt, k) for k in ['FOLDS', 'TRAIN_DAYS', 'STORE_TRAIN_DAYS', 'N_ROUNDS']})
bt.status()

---
## 2.5 Backtest: เทรนและวัดผลทุกวิธี

แต่ละรอบวัด WRMSSE ทั้งภาพรวมและรายหมวด พร้อม **bias** (ผลรวมที่ทายสูงหรือต่ำกว่ายอดจริงกี่ %)
ถ้า bias ติดลบแปลว่าทายต่ำเกินไปอย่างเป็นระบบ ซึ่งจะทำให้ของขาด

In [ ]:
rows, last_fold = [], {}
for end in FOLDS:
    train_wide = sales[ID_COLS + d_cols[:end]]
    y_true = Y[:, end:end + HORIZON]
    scopes = {'ALL': np.ones(len(sales), bool)} | {c: (sales['cat_id'] == c).to_numpy() for c in ['FOODS', 'HOUSEHOLD', 'HOBBIES']}
    scorers = {s: WRMSSE(train_wide[m], calendar, prices) for s, m in scopes.items()}

    methods = {name: (lambda f=f, end=end: (f(end), 0.0)) for name, f in BASELINES.items()}
    methods |= {name: (lambda name=name, end=end: model_forecast(name, end)) for name in MODELS}
    for name, run in methods.items():
        pred, train_min = run()
        row = {'fold': f'd_{end + 1}–d_{end + HORIZON}', 'method': name, 'train_min': train_min,
               'bias_%': (pred.sum() / y_true.sum() - 1) * 100}
        row |= {s: scorers[s].score(y_true[m], pred[m]) for s, m in scopes.items()}
        rows.append(row)
        print(f"{row['fold']}  {name:15s} WRMSSE {row['ALL']:.3f}  ({row['train_min']:.1f} min)")
        if end == FOLDS[-1]:
            last_fold[name] = pred.astype(np.float32)

backtest = pd.DataFrame(rows)

---
## 2.6 ผลลัพธ์

### 2.6.1 ภาพใหญ่: วิธีไหนแม่นที่สุด

WRMSSE เฉลี่ย 3 รอบ ยิ่งแท่งสั้นยิ่งดี เส้นสีดำคือ ± 1 std (ยิ่งสั้นยิ่งสม่ำเสมอ)

In [ ]:
comparison = (
    backtest.groupby('method')[['ALL', 'FOODS', 'HOUSEHOLD', 'HOBBIES', 'bias_%', 'train_min']].mean()
    .rename(columns={'ALL': 'WRMSSE'})
    .assign(WRMSSE_std=backtest.groupby('method')['ALL'].std(ddof=0))
    .sort_values('WRMSSE')
)
comparison = comparison[['WRMSSE', 'WRMSSE_std', 'FOODS', 'HOUSEHOLD', 'HOBBIES', 'bias_%', 'train_min']]
comparison.to_csv(ANALYSIS / 'model_comparison.csv')
backtest.to_csv(ANALYSIS / 'backtest_by_fold.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
colors = ['seagreen' if m in MODELS else 'gray' for m in comparison.index]
comparison['WRMSSE'].plot.barh(ax=axes[0], color=colors, xerr=comparison['WRMSSE_std'], capsize=3)
axes[0].invert_yaxis()
axes[0].bar_label(axes[0].containers[1], fmt='%.3f', padding=8)
axes[0].set(title=f'WRMSSE, mean of {len(FOLDS)} backtests (lower is better)', xlabel='WRMSSE', ylabel='')

per_fold = backtest.pivot(index='fold', columns='method', values='ALL')[comparison.index]
per_fold.T.plot.bar(ax=axes[1], rot=30)
axes[1].set(title='WRMSSE by method and backtest period', xlabel='', ylabel='WRMSSE')
show(fig, '02_6_model_comparison')
comparison


### 2.6.2 เลือกโมเดล

เกณฑ์: เลือกตัวที่ WRMSSE เฉลี่ย 3 รอบต่ำที่สุด ถ้ามีตัวอื่นแม่นใกล้เคียงกัน (ห่างไม่เกิน 1%) ให้เลือกตัวที่ std ต่ำกว่า เพราะผลสม่ำเสมอกว่าในทุกช่วงเวลา

In [ ]:
best_score = comparison['WRMSSE'].min()
best = comparison[comparison['WRMSSE'] <= best_score * 1.01].sort_values('WRMSSE_std').index[0]
baseline = comparison.loc[list(BASELINES), 'WRMSSE'].idxmin()
improvement = (1 - comparison.loc[best, 'WRMSSE'] / comparison.loc[baseline, 'WRMSSE']) * 100
print(f'โมเดลที่เลือก: {best} (WRMSSE {comparison.loc[best, "WRMSSE"]:.3f} ± {comparison.loc[best, "WRMSSE_std"]:.3f})')
print(f'baseline ที่ดีที่สุด: {baseline} (WRMSSE {comparison.loc[baseline, "WRMSSE"]:.3f} ± {comparison.loc[baseline, "WRMSSE_std"]:.3f})')
print(f'แม่นขึ้นจาก baseline {improvement:.1f}%')
per_fold[[best, baseline]].round(3)


### 2.6.3 Drill down: แยกตามหมวดสินค้าและ bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
comparison[['FOODS', 'HOUSEHOLD', 'HOBBIES']].plot.bar(ax=axes[0], rot=30)
axes[0].set(title='WRMSSE by category (lower is better)', xlabel='', ylabel='WRMSSE')
comparison['bias_%'].plot.bar(ax=axes[1], rot=30, color=np.where(comparison['bias_%'] > 0, 'steelblue', 'indianred'))
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set(title='Bias: total forecast vs actual (%)  — negative = under-forecast', xlabel='', ylabel='%')
show(fig, '02_6_by_category_bias')

### 2.6.4 Drill down: แยก 12 ระดับของ hierarchy (รอบ backtest ล่าสุด)

ระดับ 1 คือยอดรวมทั้งบริษัท ระดับ 12 คือรายสินค้า×สาขา ปกติระดับล่างจะยากกว่า เพราะยอดขายรายตัวผันผวนมาก

In [ ]:
LEVEL_NAMES = ['Total', 'State', 'Store', 'Category', 'Dept', 'State×Cat', 'State×Dept', 'Store×Cat',
               'Store×Dept', 'Item', 'Item×State', 'Item×Store']
end = FOLDS[-1]
y_true = Y[:, end:end + HORIZON]
scorer = WRMSSE(sales[ID_COLS + d_cols[:end]], calendar, prices)
by_level = pd.DataFrame({name: scorer.score(y_true, last_fold[name], by_level=True)[1] for name in [baseline, best]})
by_level.index = [f'{i}. {n}' for i, n in zip(by_level.index, LEVEL_NAMES)]
by_level.to_csv(ANALYSIS / 'wrmsse_by_level.csv')

fig, ax = plt.subplots(figsize=(12, 4.5))
by_level.plot.bar(ax=ax, rot=45, color=['gray', 'seagreen'])
ax.set(title=f'WRMSSE by hierarchy level, backtest d_{end + 1}–d_{end + HORIZON}', ylabel='WRMSSE')
show(fig, '02_6_by_level')

### 2.6.5 Drill down: ทายได้ใกล้ของจริงแค่ไหน (รอบ backtest ล่าสุด)

เปรียบเทียบยอดจริง (สีดำ) กับค่าที่ทาย จากภาพรวมทั้งบริษัท → รายหมวด → สินค้ารายตัว

In [ ]:
test_dates = dates[end:end + HORIZON]
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
ax_total = plt.subplot2grid((2, 3), (0, 0), colspan=3, fig=fig)
for ax in axes[0]:
    ax.remove()
ax_total.plot(test_dates, y_true.sum(axis=0), color='black', lw=2, label='Actual')
ax_total.plot(test_dates, last_fold[best].sum(axis=0), color='seagreen', lw=2, label=best)
ax_total.plot(test_dates, last_fold[baseline].sum(axis=0), color='gray', ls='--', label=baseline)
ax_total.set(title='Total company: daily units sold', ylabel='Units')
ax_total.legend()
for ax, cat in zip(axes[1], ['FOODS', 'HOUSEHOLD', 'HOBBIES']):
    m = (sales['cat_id'] == cat).to_numpy()
    ax.plot(test_dates, y_true[m].sum(axis=0), color='black', lw=2)
    ax.plot(test_dates, last_fold[best][m].sum(axis=0), color='seagreen', lw=2)
    ax.plot(test_dates, last_fold[baseline][m].sum(axis=0), color='gray', ls='--')
    ax.set(title=cat)
for ax in [ax_total, *axes[1]]:
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.SU))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
show(fig, '02_6_actual_vs_forecast_total')

In [ ]:
history_zero = (Y[:, end - 365:end] == 0).mean(axis=1)
revenue_rank = (Y[:, end - 28:end] * np.nan_to_num(fb.price[:, end - 28:end])).sum(axis=1)
picks = {
    'Fast-moving': int(np.argmax(np.where(history_zero < 0.3, revenue_rank, -1))),
    'Medium': int(np.argmax(np.where((history_zero >= 0.3) & (history_zero < 0.7), revenue_rank, -1))),
    'Slow-moving': int(np.argmax(np.where(history_zero >= 0.7, revenue_rank, -1))),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (label, i) in zip(axes, picks.items()):
    ax.bar(test_dates, y_true[i], color='lightgray', label='Actual')
    ax.plot(test_dates, last_fold[best][i], color='seagreen', lw=2, label=best)
    ax.plot(test_dates, last_fold[baseline][i], color='gray', ls='--', label=baseline)
    ax.set(title=f"{label}: {sales.loc[i, 'item_id']} @ {sales.loc[i, 'store_id']}", ylabel='Units')
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.SU))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
axes[0].legend(fontsize=8)
show(fig, '02_6_actual_vs_forecast_items')

np.savez_compressed(
    PROCESSED / 'backtest_last_fold.npz',
    end=end, best=best, baseline=baseline, y_true=y_true,
    pred_best=last_fold[best], pred_baseline=last_fold[baseline],
    price=fb.price[:, end:end + HORIZON], cat_id=sales['cat_id'].to_numpy(),
    item_id=sales['item_id'].to_numpy(), store_id=sales['store_id'].to_numpy(),
)

**แนวทางอ่านกราฟ:** เปรียบเทียบผลรวมและสินค้ารายตัวกับยอดจริงและ baseline โดยพิจารณาความคลาดเคลื่อนและ bias จากผลที่รันจริง

---
## 2.7 พยากรณ์จริง 28 วันข้างหน้า

เทรนโมเดลที่เลือกใหม่ด้วยข้อมูลถึง d_1941 (22 พ.ค. 2016) โดยใช้ช่วงประวัติเท่ากับตอน backtest แล้วพยากรณ์ d_1942–d_1969 (23 พ.ค. – 19 มิ.ย. 2016)

In [ ]:
if best in MODELS:
    final = bt.load_final(best) or bt.run_final(best, fb)  # อ่านจากไฟล์ ถ้ายังไม่มีให้เทรน
    future, model = final['pred'], final['model']
    X_future, _ = fb.build(FINAL_END + 1, FINAL_END + HORIZON)
else:
    future, model, X_future = BASELINES[best](FINAL_END), None, None

future_dates = dates[FINAL_END:FINAL_END + HORIZON]
forecast = pd.DataFrame(future, columns=pd.DatetimeIndex(future_dates).strftime('%Y-%m-%d'))
forecast.insert(0, 'store_id', sales['store_id'])
forecast.insert(0, 'item_id', sales['item_id'])
forecast.to_parquet(PROCESSED / 'forecast_next_28_days.parquet', index=False)
not_launched = np.isnan(fb.price[:, FINAL_END:FINAL_END + HORIZON])
assert future.shape == (len(sales), HORIZON) and np.isfinite(future).all() and (future >= 0).all()
assert (future[not_launched] == 0).all()  # สินค้าที่ยังไม่วางขายต้องทายเป็น 0

hist = slice(FINAL_END - 120, FINAL_END)
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(dates[hist], Y[:, hist].sum(axis=0), color='black', label='Actual (history)')
axes[0].plot(future_dates, future.sum(axis=0), color='seagreen', lw=2, label=f'Forecast ({best})')
axes[0].axvline(dates[FINAL_END], color='gray', ls=':')
axes[0].set(title='Total company: last 120 days + next 28 days forecast', ylabel='Units')
axes[0].legend()
for cat, color in zip(['FOODS', 'HOUSEHOLD', 'HOBBIES'], ['C0', 'C1', 'C2']):
    m = (sales['cat_id'] == cat).to_numpy()
    axes[1].plot(dates[hist], Y[m][:, hist].sum(axis=0), color=color, lw=1, label=cat)
    axes[1].plot(future_dates, future[m].sum(axis=0), color=color, lw=2.5, ls='--')
axes[1].axvline(dates[FINAL_END], color='gray', ls=':')
axes[1].set(title='By category (dashed = forecast)', ylabel='Units')
axes[1].legend()
show(fig, '02_7_future_forecast')

---
## 2.8 อธิบายโมเดล (Explainable AI) ด้วย SHAP

SHAP แบ่งค่าพยากรณ์ของแต่ละแถวออกเป็น "ส่วนที่มาจาก feature แต่ละตัว" ใช้ตอบ 3 คำถามสำหรับผู้ใช้ 3 กลุ่ม

| คำถาม | ผู้ใช้ | กราฟ |
|---|---|---|
| โดยรวมแล้วโมเดลใช้ปัจจัยกลุ่มไหนมากที่สุด | ผู้บริหาร | 2.8.1 ความสำคัญรายกลุ่ม |
| ทำไมสินค้าตัวนี้ วันนี้ ถึงทายเท่านี้ | ผู้จัดการสาขา | 2.8.2 waterfall รายสินค้า |
| ลดราคาแล้วโมเดลคาดว่ายอดเพิ่มแค่ไหน | ฝ่ายจัดซื้อและการตลาด | 2.8.3 ผลของราคา |

**วิธีอ่านค่า SHAP ของโมเดลนี้:** โมเดล boosting ที่ใช้ Tweedie loss ทายบน **สเกล log** ค่า SHAP จึงบวกกันบนสเกล log และมีผลแบบ **คูณ** บนจำนวนชิ้น
- ค่าพยากรณ์ = e^(ค่าฐาน + ผลรวม SHAP ทุก feature)
- feature ที่มีค่า SHAP = φ ทำให้ค่าพยากรณ์เปลี่ยน (e^φ − 1) × 100% เช่น φ = +0.10 → +10.5%, φ = −0.22 → −20%

**ข้อควรระวัง**
- SHAP บอกว่า **โมเดลใช้อะไร** ในการทาย ไม่ได้บอกเหตุและผล
- feature ที่สัมพันธ์กันเอง (เช่น lag กับ rolling mean) จะแบ่งความสำคัญกัน จึงควรดูเป็นกลุ่ม

ใช้ตัวอย่าง 20,000 แถวที่สุ่มจากช่วงพยากรณ์จริง (เฉพาะสินค้าที่วางขายแล้ว)

In [ ]:
FEATURE_GROUPS = {
    'Sales history': [c for c in X_future.columns if c.startswith(('lag_', 'rmean_'))],
    'Price': ['sell_price', 'price_rel_regular', 'price_change_7d'],
    'Calendar & events': ['wday', 'month', 'mday', 'event_name_1', 'event_type_1'],
    'SNAP': ['snap'],
    'Item & store': ID_COLS,
}


def per_store(model, X):
    """จับคู่แถวกับโมเดลที่ใช้ทาย: โมเดลเดียวใช้ทุกแถว ส่วนโมเดลรายสาขาใช้ตามรหัสสาขา"""
    if not isinstance(model, dict):
        return [(np.ones(len(X), bool), model)]
    return [((X['store_id'] == code).to_numpy(), m) for code, m in model.items()]


def explain(model, X):
    """ค่า SHAP (สเกล log) และค่าฐานของแต่ละแถว"""
    values, base = np.zeros(X.shape), np.zeros(len(X))
    for rows, m in per_store(model, X):
        if rows.any():
            explainer = shap.TreeExplainer(m)
            values[rows] = explainer.shap_values(X[rows])
            base[rows] = np.ravel(explainer.expected_value)[0]
    return values, base


LOG_LINK = best in MODELS and best != 'Random Forest'  # boosting + Tweedie ทายบนสเกล log
if LOG_LINK:
    X_shap = X_future[X_future['sell_price'].notna()].sample(20_000, random_state=42)
    shap_values, shap_base = explain(model, X_shap)
    pred_check = np.zeros(len(X_shap))
    for rows, m in per_store(model, X_shap):
        if rows.any():
            pred_check[rows] = m.predict(X_shap[rows])
    # ค่าฐาน + ผลรวม SHAP ต้องย้อนกลับเป็นค่าพยากรณ์ของโมเดลได้ (ยืนยันว่าอ่านเป็นสเกล log ถูกต้อง)
    np.testing.assert_allclose(np.exp(shap_base + shap_values.sum(axis=1)), pred_check, rtol=1e-3, atol=1e-6)
    group_shap = pd.DataFrame({g: shap_values[:, [X_shap.columns.get_loc(c) for c in cols]].sum(axis=1)
                               for g, cols in FEATURE_GROUPS.items()})
    print(f'คำนวณ SHAP {len(X_shap):,} แถว | ค่าฐาน (ยอดทายเฉลี่ยก่อนดู feature) ≈ {np.exp(shap_base.mean()):.2f} ชิ้น/วัน')
else:
    print(f'{best} ไม่ได้ทายบนสเกล log จึงข้ามการแปลง SHAP เป็น %')

### 2.8.1 ภาพรวม: ปัจจัยกลุ่มไหนสำคัญที่สุด (สำหรับผู้บริหาร)

รวมค่า SHAP ของ feature ในกลุ่มเดียวกัน แล้ววัดว่า **โดยเฉลี่ยกลุ่มนั้นทำให้ค่าพยากรณ์ขยับขึ้นหรือลงกี่ %**

In [ ]:
if LOG_LINK:
    group_importance = ((np.exp(group_shap.abs().mean()) - 1) * 100).sort_values()
    group_importance.rename('mean_abs_effect_pct').to_csv(ANALYSIS / 'shap_group_importance.csv')

    fig, ax = plt.subplots(figsize=(9, 4))
    group_importance.plot.barh(ax=ax, color='seagreen')
    ax.bar_label(ax.containers[0], fmt='±%.0f%%', padding=3)
    ax.set(title=f'{best}: average effect of each feature group on the forecast',
           xlabel='Typical change in forecast (±%)', ylabel='')
    show(fig, '02_8_shap_groups')
    display(group_importance.sort_values(ascending=False).round(1).to_frame('± % เฉลี่ย'))

รายละเอียดราย feature (beeswarm)
- **แกนตั้ง:** feature เรียงจากสำคัญมากไปน้อย
- **จุดแต่ละจุด:** 1 แถวข้อมูล สีแดงคือค่า feature สูง สีน้ำเงินคือค่าต่ำ
- **แกนนอน:** ค่า SHAP บนสเกล log ถ้าจุดอยู่ทางขวา feature นั้นดันยอดพยากรณ์ให้สูงขึ้น (+0.1 ≈ +10%)

In [ ]:
if LOG_LINK:
    shap.summary_plot(shap_values, X_shap, max_display=15, show=False, plot_size=(10, 6))
    plt.title(f'{best}: what drives the forecast (SHAP, log scale)')
    show(plt.gcf(), '02_8_shap_beeswarm')

### 2.8.2 รายสินค้า: ทำไมทายเท่านี้ (สำหรับผู้จัดการสาขา)

สินค้าที่ทำรายได้สูงสุดใน 28 วันล่าสุดของหมวด FOODS และ HOUSEHOLD ในวันที่ทายยอดสูงสุดของช่วงพยากรณ์
- กราฟ waterfall เริ่มจากค่าฐาน E[f(x)] แล้วบวกผลของแต่ละ feature จนได้ค่าพยากรณ์ f(x) (สเกล log)
- แท่งสีแดงดันยอดขึ้น แท่งสีน้ำเงินดึงยอดลง ตารางใต้กราฟแปลงผลแต่ละ feature เป็น %
- ค่าของ feature ID (เช่น item_id) ในกราฟเป็นรหัสตัวเลข ไม่ใช่ชื่อสินค้า

In [ ]:
if LOG_LINK:
    recent = slice(FINAL_END - 28, FINAL_END)
    item_revenue = (Y[:, recent] * np.nan_to_num(fb.price[:, recent])).sum(axis=1)
    for k, cat in enumerate(['FOODS', 'HOUSEHOLD'], start=1):
        i = int(np.argmax(np.where((sales['cat_id'] == cat).to_numpy(), item_revenue, -1)))
        X_item, _ = fb.build(FINAL_END + 1, FINAL_END + HORIZON, rows=[i])
        day = int(np.argmax(future[i]))
        row = X_item.iloc[[day]]
        values, base = explain(model, row)
        label = f"{sales.loc[i, 'item_id']} @ {sales.loc[i, 'store_id']}, {pd.Timestamp(future_dates[day]):%a %d %b %Y}"

        explanation = shap.Explanation(values[0], base[0], row.iloc[0].to_numpy(float), feature_names=list(row.columns))
        shap.plots.waterfall(explanation, max_display=10, show=False)
        plt.title(f'{label}: forecast {future[i, day]:.1f} units')
        show(plt.gcf(), f'02_8_shap_waterfall_{k}')

        effect = pd.Series((np.exp(values[0]) - 1) * 100, index=row.columns)
        top = effect.abs().sort_values(ascending=False).index[:8]
        display(pd.DataFrame({'ค่า feature': row.iloc[0][top], 'ผลต่อค่าพยากรณ์ (%)': effect[top].round(1)}))

### 2.8.3 ผลของราคา (สำหรับฝ่ายจัดซื้อและการตลาด)

- **แกนนอน:** ราคาเทียบราคาปกติ (`price_rel_regular`, 1.0 = ราคาปกติ, 0.8 = ลด 20%)
- **แกนตั้ง:** ผลรวมของ feature ราคาทั้ง 3 ตัวต่อค่าพยากรณ์ (%)
- **เส้น:** ค่ากลาง (median) ของแต่ละช่วงส่วนลด

In [ ]:
if LOG_LINK:
    price_effect = (np.exp(group_shap['Price'].to_numpy()) - 1) * 100
    rel = X_shap['price_rel_regular'].to_numpy()
    cat_codes = dict(zip(fb.ids['cat_id'].cat.categories, range(len(fb.ids['cat_id'].cat.categories))))
    bins = pd.IntervalIndex.from_breaks([0, 0.8, 0.9, 0.95, 0.99, 1.0], closed='right')

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
    price_table = {}
    for ax, cat in zip(axes, ['FOODS', 'HOUSEHOLD', 'HOBBIES']):
        m = (X_shap['cat_id'] == cat_codes[cat]).to_numpy()
        ax.scatter(rel[m], price_effect[m], s=4, alpha=0.25, color='steelblue')
        binned = pd.Series(price_effect[m]).groupby(pd.cut(rel[m], bins), observed=False).agg(['median', 'size'])
        price_table[cat] = binned
        ax.plot([b.mid for b in binned.index], binned['median'], color='darkorange', marker='o', lw=2)
        ax.axhline(0, color='black', lw=0.8)
        ax.set(title=cat, xlabel='Price vs regular price', xlim=(0.5, 1.02))
    axes[0].set_ylabel('Effect of price features on forecast (%)')
    show(fig, '02_8_shap_price')

    price_table = pd.concat(price_table, axis=1)
    price_table.index = ['ลด > 20%', 'ลด 10–20%', 'ลด 5–10%', 'ลด 1–5%', 'ราคาปกติ']
    price_table.to_csv(ANALYSIS / 'shap_price_effect.csv')
    display(price_table.round(1))

**สิ่งที่เห็นจาก SHAP**

XAI_FINDINGS_PLACEHOLDER

---
## 2.9 สรุปขั้นที่ 2

SUMMARY_PLACEHOLDER